# code for Appendix B and C

## T and p

In [3]:
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

torch.manual_seed(756)
np.random.seed(756)

# =========================================================
# Notebook mode
# Current notebook assumed at:
#   MFDNN_code/AppendixBC
# =========================================================
APPENDIXBC_DIR = Path.cwd().resolve()
PROJECT_ROOT = APPENDIXBC_DIR.parent
METHOD_DIR = PROJECT_ROOT / "Method"
DATA_PATH = APPENDIXBC_DIR / "Data"
RESULTS_DIR = APPENDIXBC_DIR / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

from Method.mfdnn import MFDNN, MFDNN_predict

Ts = [32, 64]
#Ts = [32, 64, 128]
Ps = [10, 50, 100]
N_TOTAL = 200
P_MAX = 100

LAM1_VALUES = [0.5, 1, 1.5, 2, 2.5, 3]
LAM2_VALUES = [0, 0.001, 0.01, 0.1, 0.5, 1]

MODEL_PARAMS = {
    "num_basis": (5, 5),
    "basis_type": "bspline",   # "bspline" or "sin"
    "degree": 3,               # used only for bspline
    "layer_sizes": [64, 64],
    "epochs": 100,
    "val_ratio": 0.2,
    "patience": 10,
}

TRUE_VARS = {0, 1}
EPSILON = 0.01


def calculate_selection_metrics(l21_norms, true_vars, epsilon=0.01):
    selected_vars = {i for i, norm in enumerate(l21_norms) if norm > epsilon}

    tp = len(selected_vars & true_vars)
    fp = len(selected_vars - true_vars)
    fn = len(true_vars - selected_vars)

    denom = 2 * tp + fp + fn
    f1_score = (2 * tp) / denom if denom > 0 else 0.0
    perfect_selection = 1.0 if selected_vars == true_vars else 0.0

    return f1_score, perfect_selection, selected_vars


def load_dataset(data_path, T, n, p_max):
    x_path = data_path / f"Xlist_T{T}_n{n}_p{p_max}.npy"
    y_path = data_path / f"ylist_T{T}_n{n}_p{p_max}.npy"

    X_array = np.load(x_path, allow_pickle=True)[0]
    y_array = np.load(y_path, allow_pickle=True)[0]
    return X_array, y_array


def select_best_hyperparameters(
    X_train,
    y_train,
    true_vars,
    p,
    domain_range,
    lam1_values,
    lam2_values,
    model_params,
    epsilon=0.01,
):
    mse_results = np.full((len(lam1_values), len(lam2_values)), np.inf)
    f1_results = np.zeros((len(lam1_values), len(lam2_values)))
    selection_info = {}

    y_train_mean = np.mean(y_train)
    y_train_std = np.std(y_train)

    for i, lam1 in enumerate(lam1_values):
        for j, lam2 in enumerate(lam2_values):
            try:
                train_losses, val_losses, model, l21 = MFDNN(
                    p=p,
                    resp=y_train,
                    func_cov=X_train,
                    num_basis=model_params["num_basis"],
                    basis_type=model_params["basis_type"],
                    degree=model_params["degree"],
                    layer_sizes=model_params["layer_sizes"],
                    domain_range=domain_range,
                    epochs=model_params["epochs"],
                    val_ratio=model_params["val_ratio"],
                    patience=model_params["patience"],
                    lam1=lam1,
                    lam2=lam2,
                    std_resp=True,
                )

                mse_val = min(val_losses) if len(val_losses) > 0 else np.mean(train_losses[-10:])
                mse_results[i, j] = mse_val

                f1_score, perfect_selection, selected_vars = calculate_selection_metrics(
                    l21, true_vars, epsilon
                )
                f1_results[i, j] = f1_score

                selection_info[f"{i}_{j}"] = {
                    "model": model,
                    "lam1": lam1,
                    "lam2": lam2,
                    "f1_score": f1_score,
                    "mse": mse_val,
                    "selected_vars": list(selected_vars),
                    "y_mean": y_train_mean,
                    "y_std": y_train_std,
                    "perfect_selection": perfect_selection,
                }

            except Exception:
                continue

    best_f1 = np.max(f1_results)
    best_indices = np.where(f1_results == best_f1)

    candidates = [
        selection_info[f"{i}_{j}"]
        for i, j in zip(best_indices[0], best_indices[1])
        if f"{i}_{j}" in selection_info
    ]

    if len(candidates) > 0:
        best_candidate = min(candidates, key=lambda x: x["mse"])
    else:
        best_candidate = {
            "model": None,
            "lam1": 1.0,
            "lam2": 0.1,
            "f1_score": 0.0,
            "mse": np.inf,
            "selected_vars": [],
            "y_mean": y_train_mean,
            "y_std": y_train_std,
            "perfect_selection": 0.0,
        }

    return best_candidate["lam1"], best_candidate["lam2"], best_candidate


def evaluate_on_test_set(best_candidate, X_test, y_test, p, domain_range, model_params):
    try:
        if best_candidate["model"] is None:
            return np.inf, best_candidate["f1_score"]

        y_mean = best_candidate["y_mean"]
        y_std = best_candidate["y_std"]

        pred_std = MFDNN_predict(
            p,
            best_candidate["model"],
            X_test,
            model_params["num_basis"],
            domain_range,
            basis_type=model_params["basis_type"],
            degree=model_params["degree"],
        )

        pred = pred_std.detach().cpu().numpy() * y_std + y_mean
        test_mse = np.mean((pred.flatten() - y_test) ** 2)
        test_rmse = np.sqrt(test_mse)

        y_test_std = np.std(y_test)
        test_nrmse = test_rmse / y_test_std if y_test_std > 0 else np.inf

        return test_nrmse, best_candidate["f1_score"]

    except Exception:
        return np.inf, best_candidate["f1_score"]


def run_tp_experiment():
    results = []

    for T in Ts:
        X_full, y_full = load_dataset(DATA_PATH, T=T, n=N_TOTAL, p_max=P_MAX)

        for p in Ps:
            start_time = time.time()
            print(f"\n{'=' * 60}")
            print(f"Processing T={T}, p={p}")
            print(f"{'=' * 60}")

            X_array = X_full[:p, :, :, :]
            p_actual, N, _, _ = X_array.shape

            split_idx = N // 2
            X_train = X_array[:, :split_idx, :, :]
            X_test = X_array[:, split_idx:, :, :]
            y_train = y_full[:split_idx]
            y_test = y_full[split_idx:]

            domain_range = [[[0, 0], [1, 1]]] * p_actual

            lam1, lam2, best_candidate = select_best_hyperparameters(
                X_train=X_train,
                y_train=y_train,
                true_vars=TRUE_VARS,
                p=p_actual,
                domain_range=domain_range,
                lam1_values=LAM1_VALUES,
                lam2_values=LAM2_VALUES,
                model_params=MODEL_PARAMS,
                epsilon=EPSILON,
            )

            test_nrmse, test_f1 = evaluate_on_test_set(
                best_candidate=best_candidate,
                X_test=X_test,
                y_test=y_test,
                p=p_actual,
                domain_range=domain_range,
                model_params=MODEL_PARAMS,
            )

            elapsed_time = time.time() - start_time
            selected_vars = set(best_candidate["selected_vars"])

            results.append(
                {
                    "T": T,
                    "p": p,
                    "lam1": lam1,
                    "lam2": lam2,
                    "test_nrmse": test_nrmse,
                    "test_f1": test_f1,
                    "selected_vars": ",".join(map(str, sorted(selected_vars))),
                    "run_time_sec": elapsed_time,
                }
            )

            print(f"Done T={T}, p={p}, time={elapsed_time:.2f}s")
            print(
                f"  lam1={lam1}, lam2={lam2}, "
                f"NRMSE={test_nrmse:.4f}, F1={test_f1:.4f}, "
                f"selected_vars={sorted(selected_vars)}"
            )

    df_results = pd.DataFrame(results).sort_values(by=["T", "p"]).reset_index(drop=True)

    df_nrmse = df_results[["T", "p", "lam1", "lam2", "test_nrmse", "test_f1", "selected_vars"]].copy()
    df_runtime = df_results[["T", "p", "run_time_sec"]].copy()

    return df_nrmse, df_runtime, df_results


df_nrmse, df_runtime, df_all = run_tp_experiment()

nrmse_path = RESULTS_DIR / "AppendixBC_mfdnn_nrmse_Tp.csv"
runtime_path = RESULTS_DIR / "AppendixBC_mfdnn_runtime_Tp.csv"

df_nrmse.to_csv(nrmse_path, index=False, encoding="utf-8-sig")
df_runtime.to_csv(runtime_path, index=False, encoding="utf-8-sig")

print(f"\nNRMSE results saved to: {nrmse_path}")
print(f"Runtime results saved to: {runtime_path}")
print("\nFull results:")
print(df_all)


Processing T=32, p=10
Done T=32, p=10, time=3.05s
  lam1=1, lam2=0, NRMSE=0.3298, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=50
Done T=32, p=50, time=10.38s
  lam1=0.5, lam2=1, NRMSE=0.3544, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=100
Done T=32, p=100, time=20.03s
  lam1=0.5, lam2=0.1, NRMSE=0.3671, F1=1.0000, selected_vars=[0, 1]

Processing T=64, p=10
Done T=64, p=10, time=11.51s
  lam1=1, lam2=0.1, NRMSE=0.3656, F1=1.0000, selected_vars=[0, 1]

Processing T=64, p=50
Done T=64, p=50, time=50.72s
  lam1=1, lam2=0, NRMSE=0.4044, F1=1.0000, selected_vars=[0, 1]

Processing T=64, p=100
Done T=64, p=100, time=100.25s
  lam1=1, lam2=0.1, NRMSE=0.4293, F1=1.0000, selected_vars=[0, 1]

NRMSE results saved to: /Users/wangdongxue/Desktop/MFDNN_STCO_revise/MFDNN_code/AppendixBC/Results/AppendixBC_mfdnn_nrmse_Tp.csv
Runtime results saved to: /Users/wangdongxue/Desktop/MFDNN_STCO_revise/MFDNN_code/AppendixBC/Results/AppendixBC_mfdnn_runtime_Tp.csv

Full results:
    T    p  

## M and n

In [5]:
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

torch.manual_seed(756)
np.random.seed(756)

# =========================================================
# Notebook mode
# Current notebook assumed at:
#   MFDNN_code/AppendixBC
# =========================================================
APPENDIXBC_DIR = Path.cwd().resolve()
PROJECT_ROOT = APPENDIXBC_DIR.parent
METHOD_DIR = PROJECT_ROOT / "Method"
DATA_PATH = APPENDIXBC_DIR / "Data"
RESULTS_DIR = APPENDIXBC_DIR / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(METHOD_DIR))

from Method.mfdnn import MFDNN, MFDNN_predict

T_FIXED = 32
P_FIXED = 10
N_MAX = 800

Ns = [200, 400, 800]
Ms = [4, 6, 8]

LAM1_VALUES = [0.5, 1, 1.5, 2, 2.5, 3]
LAM2_VALUES = [0, 0.001, 0.01, 0.1, 0.5, 1]

BASE_MODEL_PARAMS = {
    "num_basis": (5, 5),
    "basis_type": "bspline",   # "bspline" or "sin"
    "degree": 3,               # used only for bspline
    "layer_sizes": [64, 64],
    "epochs": 100,
    "val_ratio": 0.2,
    "patience": 10,
}

TRUE_VARS = {0, 1}
EPSILON = 0.01


def calculate_selection_metrics(l21_norms, true_vars, epsilon=0.01):
    selected_vars = {i for i, norm in enumerate(l21_norms) if norm > epsilon}

    tp = len(selected_vars & true_vars)
    fp = len(selected_vars - true_vars)
    fn = len(true_vars - selected_vars)

    denom = 2 * tp + fp + fn
    f1_score = (2 * tp) / denom if denom > 0 else 0.0
    perfect_selection = 1.0 if selected_vars == true_vars else 0.0

    return f1_score, perfect_selection, selected_vars


def load_dataset(data_path, T, n, p):
    x_path = data_path / f"Xlist_T{T}_n{n}_p{p}.npy"
    y_path = data_path / f"ylist_T{T}_n{n}_p{p}.npy"

    X_array = np.load(x_path, allow_pickle=True)[0]
    y_array = np.load(y_path, allow_pickle=True)[0]
    return X_array, y_array


def select_best_hyperparameters(
    X_train,
    y_train,
    true_vars,
    p,
    domain_range,
    lam1_values,
    lam2_values,
    model_params,
    epsilon=0.01,
):
    mse_results = np.full((len(lam1_values), len(lam2_values)), np.inf)
    f1_results = np.zeros((len(lam1_values), len(lam2_values)))
    selection_info = {}

    y_train_mean = np.mean(y_train)
    y_train_std = np.std(y_train)

    for i, lam1 in enumerate(lam1_values):
        for j, lam2 in enumerate(lam2_values):
            try:
                train_losses, val_losses, model, l21 = MFDNN(
                    p=p,
                    resp=y_train,
                    func_cov=X_train,
                    num_basis=model_params["num_basis"],
                    basis_type=model_params["basis_type"],
                    degree=model_params["degree"],
                    layer_sizes=model_params["layer_sizes"],
                    domain_range=domain_range,
                    epochs=model_params["epochs"],
                    val_ratio=model_params["val_ratio"],
                    patience=model_params["patience"],
                    lam1=lam1,
                    lam2=lam2,
                    std_resp=True,
                )

                mse_val = min(val_losses) if len(val_losses) > 0 else np.mean(train_losses[-10:])
                mse_results[i, j] = mse_val

                f1_score, perfect_selection, selected_vars = calculate_selection_metrics(
                    l21, true_vars, epsilon
                )
                f1_results[i, j] = f1_score

                selection_info[f"{i}_{j}"] = {
                    "model": model,
                    "lam1": lam1,
                    "lam2": lam2,
                    "f1_score": f1_score,
                    "mse": mse_val,
                    "selected_vars": list(selected_vars),
                    "y_mean": y_train_mean,
                    "y_std": y_train_std,
                    "perfect_selection": perfect_selection,
                }

            except Exception:
                continue

    best_f1 = np.max(f1_results)
    best_indices = np.where(f1_results == best_f1)

    candidates = [
        selection_info[f"{i}_{j}"]
        for i, j in zip(best_indices[0], best_indices[1])
        if f"{i}_{j}" in selection_info
    ]

    if len(candidates) > 0:
        best_candidate = min(candidates, key=lambda x: x["mse"])
    else:
        best_candidate = {
            "model": None,
            "lam1": 1.0,
            "lam2": 0.1,
            "f1_score": 0.0,
            "mse": np.inf,
            "selected_vars": [],
            "y_mean": y_train_mean,
            "y_std": y_train_std,
            "perfect_selection": 0.0,
        }

    return best_candidate["lam1"], best_candidate["lam2"], best_candidate


def evaluate_on_test_set(best_candidate, X_test, y_test, p, domain_range, model_params):
    try:
        if best_candidate["model"] is None:
            return np.inf, best_candidate["f1_score"]

        y_mean = best_candidate["y_mean"]
        y_std = best_candidate["y_std"]

        pred_std = MFDNN_predict(
            p,
            best_candidate["model"],
            X_test,
            model_params["num_basis"],
            domain_range,
            basis_type=model_params["basis_type"],
            degree=model_params["degree"],
        )

        pred = pred_std.detach().cpu().numpy() * y_std + y_mean
        test_mse = np.mean((pred.flatten() - y_test) ** 2)
        test_rmse = np.sqrt(test_mse)

        y_test_std = np.std(y_test)
        test_nrmse = test_rmse / y_test_std if y_test_std > 0 else np.inf

        return test_nrmse, best_candidate["f1_score"]

    except Exception:
        return np.inf, best_candidate["f1_score"]


def run_nM_experiment():
    X_array_full, y_array_full = load_dataset(DATA_PATH, T=T_FIXED, n=N_MAX, p=P_FIXED)

    rows = []

    for n_total in Ns:
        for M in Ms:
            start_time = time.time()
            print(f"\n{'=' * 60}")
            print(f"Processing T={T_FIXED}, p={P_FIXED}, n={n_total}, M={M}")
            print(f"{'=' * 60}")

            X_array = X_array_full[:, :n_total, :, :]
            y_array = y_array_full[:n_total]

            split_idx = n_total // 2
            X_train = X_array[:, :split_idx, :, :]
            X_test = X_array[:, split_idx:, :, :]
            y_train = y_array[:split_idx]
            y_test = y_array[split_idx:]

            model_params = BASE_MODEL_PARAMS.copy()
            model_params["num_basis"] = (M, M)

            domain_range = [[[0, 0], [1, 1]]] * P_FIXED

            lam1, lam2, best_candidate = select_best_hyperparameters(
                X_train=X_train,
                y_train=y_train,
                true_vars=TRUE_VARS,
                p=P_FIXED,
                domain_range=domain_range,
                lam1_values=LAM1_VALUES,
                lam2_values=LAM2_VALUES,
                model_params=model_params,
                epsilon=EPSILON,
            )

            test_nrmse, test_f1 = evaluate_on_test_set(
                best_candidate=best_candidate,
                X_test=X_test,
                y_test=y_test,
                p=P_FIXED,
                domain_range=domain_range,
                model_params=model_params,
            )

            elapsed_time = time.time() - start_time
            selected_vars = set(best_candidate["selected_vars"])

            rows.append(
                {
                    "T": T_FIXED,
                    "p": P_FIXED,
                    "n_total": n_total,
                    "M": M,
                    "lam1": lam1,
                    "lam2": lam2,
                    "test_nrmse": test_nrmse,
                    "test_f1": test_f1,
                    "selected_vars": ",".join(map(str, sorted(selected_vars))),
                    "run_time_sec": elapsed_time,
                }
            )

            print(f"Done T={T_FIXED}, p={P_FIXED}, n={n_total}, M={M}, time={elapsed_time:.2f}s")
            print(
                f"  lam1={lam1}, lam2={lam2}, "
                f"NRMSE={test_nrmse:.4f}, F1={test_f1:.4f}, "
                f"selected_vars={sorted(selected_vars)}"
            )

    df_results = pd.DataFrame(rows).sort_values(["n_total", "M"]).reset_index(drop=True)

    df_nrmse = df_results[
        ["T", "p", "n_total", "M", "lam1", "lam2", "test_nrmse", "test_f1", "selected_vars"]
    ].copy()
    df_runtime = df_results[
        ["T", "p", "n_total", "M", "run_time_sec"]
    ].copy()

    return df_nrmse, df_runtime, df_results


df_nrmse, df_runtime, df_all = run_nM_experiment()

nrmse_path = RESULTS_DIR / "AppendixBC_mfdnn_nrmse_nM.csv"
runtime_path = RESULTS_DIR / "AppendixBC_mfdnn_runtime_nM.csv"

df_nrmse.to_csv(nrmse_path, index=False, encoding="utf-8-sig")
df_runtime.to_csv(runtime_path, index=False, encoding="utf-8-sig")

print(f"\nNRMSE results saved to: {nrmse_path}")
print(f"Runtime results saved to: {runtime_path}")
print("\nFull results:")
print(df_all)


Processing T=32, p=10, n=200, M=4
Done T=32, p=10, n=200, M=4, time=2.11s
  lam1=1, lam2=0.01, NRMSE=0.3307, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=10, n=200, M=6
Done T=32, p=10, n=200, M=6, time=2.84s
  lam1=1, lam2=0.1, NRMSE=0.3494, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=10, n=200, M=8
Done T=32, p=10, n=200, M=8, time=5.20s
  lam1=1, lam2=0.01, NRMSE=0.3841, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=10, n=400, M=4
Done T=32, p=10, n=400, M=4, time=2.65s
  lam1=1, lam2=0.5, NRMSE=0.3465, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=10, n=400, M=6
Done T=32, p=10, n=400, M=6, time=3.62s
  lam1=1, lam2=0.001, NRMSE=0.3612, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=10, n=400, M=8
Done T=32, p=10, n=400, M=8, time=7.82s
  lam1=1, lam2=0.001, NRMSE=0.3881, F1=1.0000, selected_vars=[0, 1]

Processing T=32, p=10, n=800, M=4
Done T=32, p=10, n=800, M=4, time=3.87s
  lam1=1, lam2=0.1, NRMSE=0.3606, F1=1.0000, selected_vars=[0, 1]

Proces